In [ ]:
import pandas as pd
import numpy as np

# Load metadata
metadata = pd.read_csv('/path/to/input/metadata.tsv', sep='\t')
group_dict = metadata.groupby('Group')['SampleID'].apply(list).to_dict()
groups = ['SA', 'ACS', 'HFrEF']
data_types = ['metabolites', 'taxa', 'gmm', 'metacyc']

final_records = []

for dt in data_types:
    abundance_df = pd.read_csv(f'/path/to/input/tables/{dt}/data_transformed_{dt}.tsv', sep='\t')
    coef_df = pd.read_csv(f'/path/to/input/tables/{dt}/all_results_{dt}.tsv', sep='\t')
    
    coef_df = coef_df[(coef_df['name'] == 'Group_ord.L') &
                      (coef_df['model'] == 'abundance')]
    
    # Apply significance thresholds based on data type
    if dt == 'metabolites':
        coef_df = coef_df[coef_df['qval_individual'] < 0.05]
    else:
        coef_df = coef_df[coef_df['pval_individual'] < 0.05]

    # Filter out unmapped/unintegrated features for metacyc
    if dt == 'metacyc':
        coef_df = coef_df[coef_df['feature'] != 'UNINTEGRATED']
        coef_df = coef_df[coef_df['feature'] != 'UNMAPPED']

    coef_df = coef_df.sort_values(by='coef', ascending=False)
    display(coef_df)
    
    # Z-score normalization for numerical columns
    numeric_cols = abundance_df.columns.difference(['feature'])
    abundance_df[numeric_cols] = abundance_df[numeric_cols].apply(lambda x: (x - x.mean()) / x.std())

    coef_lookup = dict(zip(coef_df['feature'], coef_df['coef']))
    target_features = list(coef_df['feature'])
    
    for feat in target_features:
        row_data = {
            'features': feat,
            'coef': coef_lookup.get(feat),
            'type': dt
        }
        
        for grp in groups:
            samples = group_dict.get(grp, [])
            # The 'feature' column in abundance_df corresponds to SampleID
            row_data[grp] = abundance_df.loc[abundance_df['feature'].isin(samples), feat].mean()
            
        final_records.append(row_data)

# Format the final dataframe
final_df = pd.DataFrame(final_records)
final_df = final_df[['features', 'SA', 'ACS', 'HFrEF', 'coef', 'type']]

print("✅ Processing Complete.")
display(final_df)

# Export the final significant feature table
final_df.to_csv('/path/to/output/sig_table.tsv', sep='\t', index=False)

,feature,metadata,value,name,coef,null_hypothesis,stderr,pval_individual,qval_individual,pval_joint,qval_joint,error,model,N,N_not_zero
105,p.Cresol.SO4,Group_ord,.L,Group_ord.L,0.853013,0.021952,0.255181,1.386745e-03,0.040216,1.386745e-03,0.036171,NaN,abundance,169,169
47,X3.Met.His,Group_ord,.L,Group_ord.L,0.811633,0.021952,0.204612,1.661072e-04,0.010865,1.661072e-04,0.009772,NaN,abundance,169,169
39,Ind.SO4,Group_ord,.L,Group_ord.L,0.644755,0.021952,0.156790,1.104219e-04,0.008563,1.104219e-04,0.007702,NaN,abundance,169,169
104,TMAO,Group_ord,.L,Group_ord.L,0.624676,0.021952,0.184595,1.357300e-03,0.040099,1.357300e-03,0.036065,NaN,abundance,169,169
8,X1.Met.His,Group_ord,.L,Group_ord.L,0.453450,0.021952,0.078402,1.811461e-07,0.000062,1.811461e-07,0.000056,NaN,abundance,169,169
6,SDMA,Group_ord,.L,Group_ord.L,0.420943,0.021952,0.071019,1.117852e-07,0.000050,1.117852e-07,0.000045,NaN,abundance,169,169
40,C7.DC,Group_ord,.L,Group_ord.L,0.329903,0.021952,0.077576,1.235143e-04,0.009000,1.235143e-04,0.008095,NaN,abundance,169,169
21,HCys,Group_ord,.L,Group_ord.L,0.325769,0.021952,0.063316,4.788224e-06,0.000675,4.788224e-06,0.000607,NaN,abundance,169,169
114,C4,Group_ord,.L,Group_ord.L,0.324195,0.021952,0.093623,1.624394e-03,0.043816,1.624394e-03,0.039755,NaN,abundance,169,169
54,Creatinine,Group_ord,.L,Group_ord.L,0.294474,0.021952,0.072566,2.782966e-04,0.015696,2.782966e-04,0.014117,NaN,abundance,169,169


,feature,metadata,value,name,coef,null_hypothesis,stderr,pval_individual,qval_individual,pval_joint,qval_joint,error,model,N,N_not_zero
147,s__GGB2653_SGB3574,Group_ord,.L,Group_ord.L,0.894324,0.027268,0.388816,0.028961,0.604216,0.057084,0.641594,NaN,abundance,169,96
227,s__GGB3109_SGB4121,Group_ord,.L,Group_ord.L,0.850853,0.027268,0.408457,0.048446,0.666804,0.094546,0.704099,NaN,abundance,169,77
231,s__Dorea_longicatena,Group_ord,.L,Group_ord.L,-0.471644,0.027268,0.249513,0.049745,0.668323,0.097016,0.704099,NaN,abundance,169,150
199,s__Blautia_massiliensis,Group_ord,.L,Group_ord.L,-0.607368,0.027268,0.311458,0.044893,0.659375,0.087770,0.698787,NaN,abundance,169,146
124,s__Clostridia_bacterium_UC5_1_1D1,Group_ord,.L,Group_ord.L,-0.783728,0.027268,0.355297,0.025486,0.581418,0.050323,0.611806,NaN,abundance,169,103
108,s__Ruminococcus_lactaris,Group_ord,.L,Group_ord.L,-0.912789,0.027268,0.381818,0.017682,0.522439,0.035051,0.542672,NaN,abundance,169,57
56,s__Anaerobutyricum_hallii,Group_ord,.L,Group_ord.L,-1.038761,0.027268,0.380737,0.006227,0.348713,0.012415,0.353835,NaN,abundance,169,126
196,s__Bacteroides_cellulosilyticus,Group_ord,.L,Group_ord.L,-1.098374,0.027268,0.544323,0.041832,0.659375,0.081914,0.698787,NaN,abundance,169,96
157,s__GGB1495_SGB2071,Group_ord,.L,Group_ord.L,-1.145333,0.027268,0.527447,0.030858,0.615620,0.060764,0.659724,NaN,abundance,169,59
94,s__Blautia_luti,Group_ord,.L,Group_ord.L,-1.258881,0.027268,0.515876,0.015724,0.502951,0.031200,0.508113,NaN,abundance,169,66


,feature,metadata,value,name,coef,null_hypothesis,stderr,pval_individual,qval_individual,pval_joint,qval_joint,error,model,N,N_not_zero
65,starch degradation,Group_ord,.L,Group_ord.L,-0.199655,0.059789,0.126293,0.046163,0.573796,0.046163,0.466756,NaN,abundance,169,169
60,tyrosine degradation II,Group_ord,.L,Group_ord.L,-0.383116,0.059789,0.215423,0.043369,0.558751,0.084858,0.594005,NaN,abundance,169,155
50,succinate conversion to propionate,Group_ord,.L,Group_ord.L,-1.337351,0.059789,0.633983,0.030882,0.473667,0.060811,0.539530,NaN,abundance,169,79


,feature,metadata,value,name,coef,null_hypothesis,stderr,pval_individual,qval_individual,pval_joint,qval_joint,error,model,N,N_not_zero
324,PWY-6897: thiamine diphosphate salvage II,Group_ord,.L,Group_ord.L,-0.036995,0.041744,0.033300,0.027025,0.337611,0.027025,0.241583,NaN,abundance,169,169
384,PWY-5973: cis-vaccenate biosynthesis,Group_ord,.L,Group_ord.L,-0.049075,0.041744,0.043585,0.045751,0.477249,0.045751,0.358360,NaN,abundance,169,169
272,PWY-6518: bile acids epimerization,Group_ord,.L,Group_ord.L,-0.565098,0.041744,0.237476,0.012744,0.189094,0.025326,0.230222,NaN,abundance,169,83


✅ Processing Complete.


,features,SA,ACS,HFrEF,coef,type
0,p.Cresol.SO4,-0.248188,-0.025542,0.286114,0.853013,metabolites
1,X3.Met.His,-0.285594,-0.134457,0.430549,0.811633,metabolites
2,Ind.SO4,-0.275847,-0.086000,0.373553,0.644755,metabolites
3,TMAO,-0.163693,-0.194888,0.360390,0.624676,metabolites
4,X1.Met.His,-0.314748,-0.293549,0.614675,0.453450,metabolites
5,SDMA,-0.217525,-0.414921,0.629281,0.420943,metabolites
6,C7.DC,-0.363517,0.044707,0.339880,0.329903,metabolites
7,HCys,-0.359442,-0.059797,0.436359,0.325769,metabolites
8,C4,-0.245165,-0.043378,0.300128,0.324195,metabolites
9,Creatinine,-0.145612,-0.313170,0.455397,0.294474,metabolites
